# Projet Home Credit – Sujet 03

**Séance :** 02  
**Nom :** ADOUKO Bismark Wafik

**Étapes :**
- Le plancher de contrôle (règle métier sans modèle)
- Le tableau d'essais (journal de tous les essais, réussis ou ratés)

**Ce que fait ce carnet**

1. Il découpe `train.csv` comme le carnet 02 : 80 % entraînement, 20 % validation, stratifié, `random_state=42`.
2. Il teste **9 règles simples** qu'un analyste appliquerait sans modèle. Le seuil de chaque règle est choisi sur l'entraînement, puis mesuré sur la validation.
3. Il retient la meilleure règle comme **plancher de contrôle** : tout modèle devra faire mieux.
4. Il écrit le **tableau d'essais** (`sorties/journal_essais.csv`), où l'équipe ajoutera ses modèles.

## 0. Réglages communs à l'équipe

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score

# Mêmes réglages que le carnet 02 (personne 2) : tout le monde compare les mêmes lignes
COUT_FN = 5.0       # défaut accordé : on perd l'argent prêté
COUT_FP = 1.0       # bon client refusé : on perd la marge, le client va ailleurs
SEED = 42
TEST_SIZE = 0.2
PORTEUR = "ADOUKO Bismark Wafik"

## 1. Charger les données

In [2]:
# Cherche train.csv sur Kaggle, puis en local
pistes = [Path("/kaggle/input"), Path.cwd(), Path.cwd() / "Sujet_03_Home_Credit",
          Path.cwd().parent / "Sujet_03_Home_Credit"]
chemin = None
for p in pistes:
    if p.exists():
        chemin = next(p.rglob("train.csv"), None) if p == Path("/kaggle/input") else (p / "train.csv" if (p / "train.csv").exists() else None)
        if chemin:
            break
dossier = chemin.parent
print("Données trouvées dans :", dossier)

train = pd.read_csv(dossier / "train.csv")
print("train :", train.shape)

# Où écrire les sorties (Kaggle : /kaggle/working, en local : ./sorties)
SORTIES = Path("/kaggle/working/sorties") if Path("/kaggle/working").exists() else Path("sorties")
SORTIES.mkdir(parents=True, exist_ok=True)

Données trouvées dans : /home/wafik/D-p-t-de-groupe-6/Sujet_03_Home_Credit


train : (246008, 122)


## 2. Le découpage de validation

On sépare `train.csv` en deux. Les seuils sont choisis **uniquement sur l'entraînement**. La validation sert seulement à mesurer, comme si c'était `a_decider.csv`.

Le découpage est **stratifié** : il garde 8 % de défauts des deux côtés.

In [3]:
y = train["TARGET"]
X = train.drop(columns=["TARGET"])

X_ent, X_val, y_ent, y_val = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

print("Entraînement :", len(X_ent), "lignes |", round(y_ent.mean() * 100, 2), "% de défauts")
print("Validation   :", len(X_val), "lignes |", round(y_val.mean() * 100, 2), "% de défauts")

Entraînement : 196806 lignes | 8.07 % de défauts
Validation   : 49202 lignes | 8.07 % de défauts


## 3. Les outils de mesure

- `mesurer` : à partir des refus, calcule la matrice de confusion, le rappel, la précision, le taux de refus et le **coût moyen par dossier**.
- `choisir_seuil` : essaie 99 seuils (les centiles du score) **sur l'entraînement** et garde celui qui coûte le moins.

Convention : un **score élevé = client risqué**, donc on refuse quand `score >= seuil`.

In [4]:
def mesurer(y_vrai, refus, cout_fn=COUT_FN, cout_fp=COUT_FP):
    """refus = 1 si on refuse le crédit, 0 si on l'accorde."""
    tn, fp, fn, tp = confusion_matrix(y_vrai, refus, labels=[0, 1]).ravel()
    n = len(y_vrai)
    return {
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "taux_refus": (tp + fp) / n,
        "rappel": tp / (tp + fn) if (tp + fn) else 0.0,       # part des défauts attrapés
        "precision": tp / (tp + fp) if (tp + fp) else 0.0,    # quand on refuse, a-t-on raison ?
        "cout_moyen": (cout_fn * fn + cout_fp * fp) / n,
    }


def choisir_seuil(score_ent, y_ent, cout_fn=COUT_FN, cout_fp=COUT_FP):
    """Essaie les centiles 1 à 99 du score d'entraînement, garde le seuil le moins coûteux."""
    candidats = np.unique(np.nanpercentile(score_ent, np.arange(1, 100)))
    couts = [mesurer(y_ent, (score_ent >= s).astype(int), cout_fn, cout_fp)["cout_moyen"] for s in candidats]
    return candidats[int(np.argmin(couts))]

print("Outils prêts.")

Outils prêts.


## 4. Les variables des règles

Uniquement des informations **connues au moment de la demande** (règle « pas de fuite ») :

| Variable | Calcul | Pourquoi un analyste la regarde |
|---|---|---|
| `ext_moyenne` | moyenne des `EXT_SOURCE_1/2/3` disponibles | scores externes, les plus prédictifs selon l'énoncé |
| `taux_effort` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | part du revenu qui part dans la mensualité |
| `age` | `-DAYS_BIRTH / 365` | les jeunes emprunteurs remboursent statistiquement moins bien |
| `anciennete_emploi` | `-DAYS_EMPLOYED / 365` | stabilité de l'emploi |

**Deux pièges traités ici :**
- `DAYS_EMPLOYED = 365243` (environ 1 000 ans) n'est pas une vraie valeur : c'est le code des **retraités et sans emploi** (18 % des lignes). On le remplace par un manque, et on ne pénalise pas ces clients.
- Les valeurs manquantes sont comblées avec la **médiane de l'entraînement**, jamais avec celle de la validation.

In [5]:
def variables_regles(df):
    v = pd.DataFrame(index=df.index)
    v["ext_moyenne"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)   # ignore les manques
    v["ext_source_2"] = df["EXT_SOURCE_2"]
    v["taux_effort"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    v["age"] = -df["DAYS_BIRTH"] / 365
    emploi = df["DAYS_EMPLOYED"].where(df["DAYS_EMPLOYED"] != 365243)              # 365243 = retraité / sans emploi
    v["anciennete_emploi"] = -emploi / 365
    v["sans_emploi_code"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    return v

V_ent = variables_regles(X_ent)
V_val = variables_regles(X_val)

# Médianes apprises sur l'entraînement seulement
medianes = V_ent.median()
V_ent = V_ent.fillna(medianes)
V_val = V_val.fillna(medianes)

# Les retraités (code 365243) : on ne les compte pas comme « ancienneté courte »
V_ent.loc[V_ent["sans_emploi_code"] == 1, "anciennete_emploi"] = V_ent["anciennete_emploi"].max()
V_val.loc[V_val["sans_emploi_code"] == 1, "anciennete_emploi"] = V_ent["anciennete_emploi"].max()

V_ent.describe().round(3)

,ext_moyenne,ext_source_2,taux_effort,age,anciennete_emploi,sans_emploi_code
count,196806.000,196806.000,196806.000,196806.000,196806.000,196806.000
mean,0.509,0.514,0.181,43.924,14.188,0.180
std,0.150,0.191,0.095,11.954,17.337,0.384
min,0.000,0.000,0.003,21.022,-0.000,0.000
25%,0.413,0.393,0.115,33.984,2.562,0.000
50%,0.524,0.566,0.163,43.151,6.090,0.000
75%,0.622,0.663,0.229,53.907,15.622,0.000
max,0.879,0.855,1.571,69.044,49.074,1.000


## 5. Les règles métier testées

Chaque règle est un **score** (élevé = risqué). Pour une variable où *bas = risqué* (score externe, âge), on prend son opposé.

La règle 9 est une **grille de points**, comme celles des banques : 1 point par critère de risque, et on refuse à partir d'un certain nombre de points. Les seuils de chaque critère sont les seuils trouvés pour les règles 4, 6, 7 et 8.

In [6]:
regles = {
    "R01": ("Tout accorder (personne n'est refusé)", [], lambda V: pd.Series(0.0, index=V.index)),
    "R02": ("Tout refuser", [], lambda V: pd.Series(1.0, index=V.index)),
    "R03": ("Refuser au hasard 8 % des dossiers", [], None),
    "R04": ("Score externe moyen bas", ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], lambda V: -V["ext_moyenne"]),
    "R05": ("EXT_SOURCE_2 bas (un seul score)", ["EXT_SOURCE_2"], lambda V: -V["ext_source_2"]),
    "R06": ("Taux d'effort élevé (annuité / revenu)", ["AMT_ANNUITY", "AMT_INCOME_TOTAL"], lambda V: V["taux_effort"]),
    "R07": ("Emprunteur jeune", ["DAYS_BIRTH"], lambda V: -V["age"]),
    "R08": ("Ancienneté d'emploi courte", ["DAYS_EMPLOYED"], lambda V: -V["anciennete_emploi"]),
}

# Pour afficher les seuils en clair : (variable, unité, sens)
EN_CLAIR = {
    "R04": ("score externe moyen", "", "<"), "R05": ("EXT_SOURCE_2", "", "<"),
    "R06": ("taux d'effort", "", ">="), "R07": ("âge", " ans", "<"),
    "R08": ("ancienneté d'emploi", " ans", "<"),
}

def seuil_en_clair(code_r, s):
    nom, unite, sens = EN_CLAIR[code_r]
    valeur = -s if sens == "<" else s          # le score est l'opposé de la variable quand « bas = risqué »
    signe = "<=" if sens == "<" else ">="
    return f"refus si {nom} {signe} {valeur:.3f}{unite}"

resultats = {}
seuils = {}
rng = np.random.default_rng(SEED)

for code_r, (nom, variables, fonction) in regles.items():
    if code_r == "R03":
        refus_val = (rng.random(len(y_val)) < y_ent.mean()).astype(int)
        auc = 0.5
        seuil_txt = "hasard 8 %"
    elif fonction(V_ent).nunique() == 1:              # tout accorder / tout refuser : pas de seuil
        refus_val = fonction(V_val).astype(int).values
        auc = 0.5
        seuil_txt = "aucun"
    else:
        score_ent, score_val = fonction(V_ent), fonction(V_val)
        s = choisir_seuil(score_ent.values, y_ent.values)
        seuils[code_r] = s
        refus_val = (score_val >= s).astype(int).values
        auc = roc_auc_score(y_val, score_val)
        seuil_txt = seuil_en_clair(code_r, s)
    m = mesurer(y_val, refus_val)
    m.update({"regle": nom, "variables": ", ".join(variables) or "aucune", "seuil": seuil_txt, "AUC_valid": auc})
    resultats[code_r] = m

print("8 règles mesurées.")

8 règles mesurées.


In [7]:
# R09 — Grille de points : 1 point par critère de risque, seuils repris de R04, R06, R07, R08
def points(V):
    return ((-V["ext_moyenne"] >= seuils["R04"]).astype(int)
            + (V["taux_effort"] >= seuils["R06"]).astype(int)
            + (-V["age"] >= seuils["R07"]).astype(int)
            + (-V["anciennete_emploi"] >= seuils["R08"]).astype(int))

pts_ent, pts_val = points(V_ent), points(V_val)
print("Répartition des points (entraînement) :")
print(pd.crosstab(pts_ent, y_ent, normalize="index").round(3).rename(columns={0: "remboursé", 1: "défaut"}))

# Combien de points pour refuser ? On choisit sur l'entraînement
meilleur_k = min(range(1, 5), key=lambda k: mesurer(y_ent, (pts_ent >= k).astype(int))["cout_moyen"])
m = mesurer(y_val, (pts_val >= meilleur_k).astype(int))
m.update({"regle": "Grille de points (score externe, effort, âge, emploi)",
          "variables": "EXT_SOURCE_1/2/3, AMT_ANNUITY, AMT_INCOME_TOTAL, DAYS_BIRTH, DAYS_EMPLOYED",
          "seuil": f"refus si points >= {meilleur_k}", "AUC_valid": roc_auc_score(y_val, pts_val)})
resultats["R09"] = m
print("Refus à partir de", meilleur_k, "point(s)")

Répartition des points (entraînement) :
TARGET  remboursé  défaut
row_0                    
0           0.939   0.061
1           0.803   0.197
2           0.798   0.202
3           0.840   0.160


Refus à partir de 1 point(s)


## 6. Résultats sur la validation

In [8]:
res = pd.DataFrame(resultats).T
res = res[["regle", "seuil", "AUC_valid", "taux_refus", "rappel", "precision", "FN", "FP", "cout_moyen"]]
cout_tout_accorder = res.loc["R01", "cout_moyen"]
res["gain_vs_tout_accorder_%"] = (1 - res["cout_moyen"].astype(float) / cout_tout_accorder) * 100

affichage = res.copy()
for c in ["taux_refus", "rappel", "precision"]:
    affichage[c] = (affichage[c].astype(float) * 100).round(1).astype(str) + " %"
affichage["AUC_valid"] = affichage["AUC_valid"].astype(float).round(3)
affichage["cout_moyen"] = affichage["cout_moyen"].astype(float).round(4)
affichage["gain_vs_tout_accorder_%"] = affichage["gain_vs_tout_accorder_%"].astype(float).round(1)
affichage.sort_values("cout_moyen")

,regle,seuil,AUC_valid,taux_refus,rappel,precision,FN,FP,cout_moyen,gain_vs_tout_accorder_%
R04,Score externe moyen bas,refus si score externe moyen <= 0.324,0.712,11.9 %,31.8 %,21.5 %,2709,4610,0.3690,8.6
R09,"Grille de points (score externe, effort, âge, ...",refus si points >= 1,0.608,14.3 %,34.2 %,19.3 %,2615,5659,0.3808,5.7
R05,EXT_SOURCE_2 bas (un seul score),refus si EXT_SOURCE_2 <= 0.155,0.653,5.9 %,15.8 %,21.6 %,3343,2287,0.3862,4.3
R01,Tout accorder (personne n'est refusé),aucun,0.500,0.0 %,0.0 %,0.0 %,3972,0,0.4036,0.0
R07,Emprunteur jeune,refus si âge <= 22.625 ans,0.581,0.9 %,1.4 %,12.1 %,3916,405,0.4062,-0.6
R08,Ancienneté d'emploi courte,refus si ancienneté d'emploi <= 0.318 ans,0.586,1.1 %,1.6 %,12.0 %,3909,463,0.4067,-0.7
R06,Taux d'effort élevé (annuité / revenu),refus si taux d'effort >= 0.485,0.524,0.9 %,0.8 %,6.6 %,3942,427,0.4093,-1.4
R03,Refuser au hasard 8 % des dossiers,hasard 8 %,0.500,8.0 %,8.4 %,8.4 %,3640,3616,0.4434,-9.8
R02,Tout refuser,aucun,0.500,100.0 %,100.0 %,8.1 %,0,45230,0.9193,-127.7


## 7. Le plancher de contrôle retenu

In [9]:
regles_metier = res.drop(index=["R01", "R02", "R03"])       # les vraies règles d'analyste
code_plancher = regles_metier["cout_moyen"].astype(float).idxmin()
PLANCHER = float(res.loc[code_plancher, "cout_moyen"])

print("Référence « tout accorder » :", round(cout_tout_accorder, 4), "par dossier")
print("PLANCHER DE CONTRÔLE        :", code_plancher, "—", res.loc[code_plancher, "regle"])
print("   seuil                    :", res.loc[code_plancher, "seuil"])
print("   coût moyen               :", round(PLANCHER, 4), "par dossier")
print("   refuse                   :", round(float(res.loc[code_plancher, "taux_refus"]) * 100, 1), "% des demandes")
print("   attrape                  :", round(float(res.loc[code_plancher, "rappel"]) * 100, 1), "% des défauts")
print()
print("=> Un modèle n'est utile que s'il fait un coût moyen INFÉRIEUR à", round(PLANCHER, 4))

# On enregistre le plancher pour que toute l'équipe compare au même chiffre
pd.Series({"regle": res.loc[code_plancher, "regle"], "seuil": res.loc[code_plancher, "seuil"],
           "cout_moyen": round(PLANCHER, 4), "cout_tout_accorder": round(cout_tout_accorder, 4),
           "COUT_FN": COUT_FN, "COUT_FP": COUT_FP, "SEED": SEED, "TEST_SIZE": TEST_SIZE}
          ).to_json(SORTIES / "plancher.json", force_ascii=False, indent=1)
print("Plancher enregistré :", SORTIES / "plancher.json")

Référence « tout accorder » : 0.4036 par dossier
PLANCHER DE CONTRÔLE        : R04 — Score externe moyen bas
   seuil                    : refus si score externe moyen <= 0.324
   coût moyen               : 0.369 par dossier
   refuse                   : 11.9 % des demandes
   attrape                  : 31.8 % des défauts

=> Un modèle n'est utile que s'il fait un coût moyen INFÉRIEUR à 0.369
Plancher enregistré : sorties/plancher.json


## 8. Le plancher est-il stable ?

Un seul découpage peut avoir de la chance. On refait la règle retenue sur **5 découpages différents** (validation croisée stratifiée en 5 morceaux). À chaque fois, le seuil est rechoisi sur la partie entraînement.

In [10]:
fonction_plancher = regles[code_plancher][2] if code_plancher in regles else None
V_tout = variables_regles(X)
couts_cv, seuils_cv = [], []
for i, (idx_e, idx_v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y)):
    Ve, Vv = V_tout.iloc[idx_e], V_tout.iloc[idx_v]
    med = Ve.median()
    Ve, Vv = Ve.fillna(med), Vv.fillna(med)
    s = choisir_seuil(fonction_plancher(Ve).values, y.iloc[idx_e].values)
    couts_cv.append(mesurer(y.iloc[idx_v], (fonction_plancher(Vv) >= s).astype(int))["cout_moyen"])
    seuils_cv.append(s)

print("Coût moyen sur les 5 morceaux :", np.round(couts_cv, 4))
print(f"=> plancher = {np.mean(couts_cv):.4f} ± {np.std(couts_cv):.4f}")
print("Seuils choisis (refus si score externe moyen <=) :", np.round(-np.array(seuils_cv), 3))

Coût moyen sur les 5 morceaux : [0.3667 0.3681 0.3645 0.3603 0.3697]
=> plancher = 0.3659 ± 0.0033
Seuils choisis (refus si score externe moyen <=) : [0.324 0.333 0.333 0.333 0.333]


## 9. Et si le rapport de coûts était différent ?

Le rapport 5 pour 1 est une **hypothèse**. On regarde comment la règle retenue réagit à d'autres rapports : le seuil change, et donc le nombre de refus.

In [11]:
sensibilite = []
score_e, score_v = fonction_plancher(V_ent).values, fonction_plancher(V_val).values
for rapport in [2, 3, 5, 8, 10]:
    s = choisir_seuil(score_e, y_ent.values, cout_fn=rapport, cout_fp=1)
    m = mesurer(y_val, (score_v >= s).astype(int), cout_fn=rapport, cout_fp=1)
    accorder = mesurer(y_val, np.zeros(len(y_val), dtype=int), cout_fn=rapport, cout_fp=1)["cout_moyen"]
    sensibilite.append({"rapport FN:FP": f"{rapport}:1", "taux_refus_%": round(m["taux_refus"] * 100, 1),
                        "rappel_%": round(m["rappel"] * 100, 1), "cout_regle": round(m["cout_moyen"], 4),
                        "cout_tout_accorder": round(accorder, 4),
                        "gain_%": round((1 - m["cout_moyen"] / accorder) * 100, 1)})
pd.DataFrame(sensibilite)

,rapport FN:FP,taux_refus_%,rappel_%,cout_regle,cout_tout_accorder,gain_%
0,2:1,1.0,4.3,0.1611,0.1615,0.2
1,3:1,2.0,7.8,0.2371,0.2422,2.1
2,5:1,11.9,31.8,0.3690,0.4036,8.6
3,8:1,21.8,48.6,0.5106,0.6458,20.9
4,10:1,28.6,57.5,0.5833,0.8073,27.7


## 10. Le tableau d'essais

Le tableau d'essais est le **journal** de l'équipe : chaque essai, **réussi ou raté**, y est noté avec son score. L'énoncé demande au moins 8 essais, y compris ceux qui ont échoué.

- Il est enregistré dans `sorties/journal_essais.csv`.
- Les règles de ce carnet en sont les premières lignes.
- Le **verdict** compare chaque essai au plancher de contrôle.

In [12]:
COLONNES_JOURNAL = ["id", "date", "porteur", "type", "description", "variables", "seuil",
                    "AUC_valid", "taux_refus_%", "rappel_%", "precision_%", "cout_moyen",
                    "ecart_vs_plancher_%", "verdict", "commentaire"]

def verdict(cout, plancher=PLANCHER, ref=cout_tout_accorder, id_essai=None):
    if id_essai == "R01":
        return "référence (tout accorder)"
    if cout >= ref:
        return "échoué (pas mieux que tout accorder)"
    if cout > plancher:
        return "échoué (moins bon que le plancher)"
    if cout == plancher:
        return "plancher de contrôle"
    return "réussi (bat le plancher)"

commentaires = {
    "R01": "Référence : aucun défaut évité, tous les défauts sont payés.",
    "R02": "Plus aucun client : on paie tous les bons clients perdus.",
    "R03": "Contrôle : refuser sans regarder le dossier n'apporte rien.",
    "R04": "Les trois scores externes ensemble classent bien les dossiers.",
    "R05": "Un seul score externe : moins bon que la moyenne des trois.",
    "R06": "Le taux d'effort seul ne sépare presque pas les défauts.",
    "R07": "L'âge seul est un signal faible.",
    "R08": "L'ancienneté seule est un signal faible.",
    "R09": "Mélanger des critères faibles dilue le score externe.",
}

lignes = []
for code_r, r in res.iterrows():
    cout = float(r["cout_moyen"])
    lignes.append({
        "id": code_r, "date": str(date.today()), "porteur": PORTEUR, "type": "règle métier",
        "description": r["regle"], "variables": resultats[code_r]["variables"], "seuil": r["seuil"],
        "AUC_valid": round(float(r["AUC_valid"]), 4),
        "taux_refus_%": round(float(r["taux_refus"]) * 100, 1),
        "rappel_%": round(float(r["rappel"]) * 100, 1),
        "precision_%": round(float(r["precision"]) * 100, 1),
        "cout_moyen": round(cout, 4),
        "ecart_vs_plancher_%": round((cout / PLANCHER - 1) * 100, 1),
        "verdict": verdict(cout, id_essai=code_r), "commentaire": commentaires.get(code_r, ""),
    })

journal = pd.DataFrame(lignes, columns=COLONNES_JOURNAL)
journal.to_csv(SORTIES / "journal_essais.csv", index=False)
print("Journal écrit :", SORTIES / "journal_essais.csv", "|", len(journal), "essais")
journal[["id", "description", "AUC_valid", "taux_refus_%", "rappel_%", "cout_moyen", "ecart_vs_plancher_%", "verdict"]]

Journal écrit : sorties/journal_essais.csv | 9 essais


,id,description,AUC_valid,taux_refus_%,rappel_%,cout_moyen,ecart_vs_plancher_%,verdict
0,R01,Tout accorder (personne n'est refusé),0.5000,0.0,0.0,0.4036,9.4,référence (tout accorder)
1,R02,Tout refuser,0.5000,100.0,100.0,0.9193,149.1,échoué (pas mieux que tout accorder)
2,R03,Refuser au hasard 8 % des dossiers,0.5000,8.0,8.4,0.4434,20.2,échoué (pas mieux que tout accorder)
3,R04,Score externe moyen bas,0.7123,11.9,31.8,0.3690,0.0,plancher de contrôle
4,R05,EXT_SOURCE_2 bas (un seul score),0.6531,5.9,15.8,0.3862,4.7,échoué (moins bon que le plancher)
5,R06,Taux d'effort élevé (annuité / revenu),0.5245,0.9,0.8,0.4093,10.9,échoué (pas mieux que tout accorder)
6,R07,Emprunteur jeune,0.5813,0.9,1.4,0.4062,10.1,échoué (pas mieux que tout accorder)
7,R08,Ancienneté d'emploi courte,0.5855,1.1,1.6,0.4067,10.2,échoué (pas mieux que tout accorder)
8,R09,"Grille de points (score externe, effort, âge, ...",0.6082,14.3,34.2,0.3808,3.2,échoué (moins bon que le plancher)


### Comment ajouter un essai (pour toute l'équipe)

Après avoir entraîné un modèle, on lui donne les **probabilités sur la même validation** (`X_val`) et le seuil choisi **sur l'entraînement**. La fonction calcule tout et ajoute une ligne au journal.

In [13]:
def ajouter_essai(id_essai, porteur, description, variables, proba_val, seuil, commentaire="", type_essai="modèle"):
    """Ajoute un essai au journal. proba_val = probabilités de défaut sur X_val (mêmes lignes, même ordre)."""
    proba_val = np.asarray(proba_val)
    m = mesurer(y_val, (proba_val >= seuil).astype(int))
    ligne = {
        "id": id_essai, "date": str(date.today()), "porteur": porteur, "type": type_essai,
        "description": description, "variables": variables, "seuil": f"proba >= {seuil:.3f}",
        "AUC_valid": round(roc_auc_score(y_val, proba_val), 4),
        "taux_refus_%": round(m["taux_refus"] * 100, 1), "rappel_%": round(m["rappel"] * 100, 1),
        "precision_%": round(m["precision"] * 100, 1), "cout_moyen": round(m["cout_moyen"], 4),
        "ecart_vs_plancher_%": round((m["cout_moyen"] / PLANCHER - 1) * 100, 1),
        "verdict": verdict(m["cout_moyen"]), "commentaire": commentaire,
    }
    chemin = SORTIES / "journal_essais.csv"
    ancien = pd.read_csv(chemin) if chemin.exists() else pd.DataFrame(columns=COLONNES_JOURNAL)
    ancien = ancien[ancien["id"] != id_essai]                     # ré-exécuter la cellule ne crée pas de doublon
    nouveau = pd.concat([ancien, pd.DataFrame([ligne])], ignore_index=True)
    nouveau.to_csv(chemin, index=False)
    return pd.DataFrame([ligne])

# Exemple d'utilisation (à décommenter par la personne qui entraîne le modèle) :
# from sklearn.linear_model import LogisticRegression
# modele = LogisticRegression(max_iter=1000).fit(X_ent_prepare, y_ent)
# proba_ent = modele.predict_proba(X_ent_prepare)[:, 1]
# proba_val = modele.predict_proba(X_val_prepare)[:, 1]
# seuil = choisir_seuil(proba_ent, y_ent.values)          # seuil choisi sur l'entraînement
# ajouter_essai("M01", "Prénom NOM", "Régression logistique", "toutes", proba_val, seuil, "premier modèle")
print("Fonction ajouter_essai prête.")

Fonction ajouter_essai prête.


## 11. Conclusions

**Le plancher de contrôle est la règle R04 : refuser quand la moyenne des scores externes (`EXT_SOURCE_1/2/3`) est inférieure ou égale à 0,324.**

| | Tout accorder (R01) | Plancher R04 |
|---|---|---|
| Demandes refusées | 0 % | 11,9 % |
| Défauts attrapés (rappel) | 0 % | 31,8 % |
| Coût moyen par dossier (5 FN + 1 FP) | 0,4036 | **0,3690** |

1. **R04 réduit le coût de 8,6 %** par rapport à « tout accorder ». Sur 5 découpages, son coût reste à 0,366 ± 0,003 : le plancher est stable.
2. **Tout modèle doit faire un coût moyen inférieur à 0,369** sur la validation pour être utile. C'est le chiffre à battre, enregistré dans `sorties/plancher.json`.
3. **Les règles sur une seule variable « classique » échouent** : le taux d'effort (R06), l'âge (R07) et l'ancienneté d'emploi (R08) font *pire* que tout accorder sur la validation. Leur signal est trop faible pour couvrir le coût des bons clients refusés.
4. **La grille de points (R09) est moins bonne que R04 seule** (0,381 contre 0,369). Ajouter des critères faibles au score externe fait refuser plus de bons clients.
5. **Refuser au hasard (R03) coûte plus cher que tout accorder** : la décision n'a de valeur que si elle s'appuie sur le dossier.
6. **Le rapport de coûts change tout.** À 2 pour 1, presque aucune règle ne vaut la peine (0,2 % de gain). À 10 pour 1, R04 refuse 28,6 % des dossiers et gagne 27,7 %. Le choix 5 pour 1 doit donc être justifié dans la note.

**Pour l'équipe** : utiliser `ajouter_essai(...)` pour chaque nouveau modèle, avec le seuil choisi sur l'entraînement. Le journal compte déjà 9 essais (dont 7 échoués), l'énoncé en demande au moins 8.